In [ ]:
%md
##Ejercicio 3.1: groupBy + agg —

In [33]:
from pyspark.sql.functions import col,count,avg,min,max

df_fact_props=spark.table("bootcamp.gold.fact_propiedades")



df=(
    df_fact_props
    .groupBy(col("zona_id"))
    .agg(
        count("*").alias("total"),
        avg(col("precio")).alias("avg_precio"),
        min(col("precio")).alias("min_precio"),
        max(col("precio")).alias("max_precio")
        )   
    .orderBy(col("total").desc())
    .limit(5)
)


df.explain(True)

#df.show()


== Parsed Logical Plan ==
'GlobalLimit 5
+- 'LocalLimit 5
   +- 'Sort ['total DESC NULLS LAST], true
      +- 'Aggregate ['zona_id], ['zona_id, 'count(*) AS total#12368, 'avg('precio) AS avg_precio#12369, 'min('precio) AS min_precio#12370, 'max('precio) AS max_precio#12371]
         +- 'UnresolvedRelation [bootcamp, gold, fact_propiedades], [], false

== Analyzed Logical Plan ==
zona_id: bigint, total: bigint, avg_precio: decimal(19,6), min_precio: decimal(15,2), max_precio: decimal(15,2)
GlobalLimit 5
+- LocalLimit 5
   +- Sort [total#12368L DESC NULLS LAST], true
      +- Aggregate [zona_id#12387L], [zona_id#12387L, count(1) AS total#12368L, avg(precio#12393) AS avg_precio#12369, min(precio#12393) AS min_precio#12370, max(precio#12393) AS max_precio#12371]
         +- SubqueryAlias bootcamp.gold.fact_propiedades
            +- Relation bootcamp.gold.fact_propiedades[row_hash#12386,zona_id#12387L,tipo_orientacion_id#12388L,fecha_id#12389L,caracteristicas_id#12390L,tipo_operacion_id#12

In [ ]:
# E3.2 JOIN SIMPLE
from pyspark.sql.functions import col

df_fact_props=spark.table("bootcamp.gold.fact_propiedades")

df_zona=spark.table("bootcamp.gold.dim_zona")

df_result=(

    df_fact_props
    .join(df_zona, df_fact_props.zona_id==df_zona.zona_id, how="inner")
    .select("partido", "region","precio","precio_por_m2")
    .limit(10)
)

df_result.explain(True)

# df_result.show()

+-------+--------------+------+-------------+
|partido|        region|precio|precio_por_m2|
+-------+--------------+------+-------------+
| moreno|gba zona oeste|400.00|        11.43|
| ezeiza|  gba zona sur|400.00|        13.33|
|  tigre|gba zona norte|400.00|        14.81|
|  tigre|gba zona norte|400.00|        14.81|
|  tigre|gba zona norte|400.00|        14.81|
|  tigre|gba zona norte|400.00|         9.52|
|  tigre|gba zona norte|400.00|         7.02|
|  tigre|gba zona norte|400.00|        14.81|
|  tigre|gba zona norte|400.00|        13.33|
|escobar|gba zona norte|400.00|         9.09|
+-------+--------------+------+-------------+



In [12]:
# E3.3. JOIN STAR SCHEMA
from pyspark.sql.functions import col, count, avg, desc

dffp=spark.table("bootcamp.gold.fact_propiedades")
dfz=spark.table("bootcamp.gold.dim_zona")
dfto=spark.table("bootcamp.gold.dim_tipo_operacion")
dfc=spark.table("bootcamp.gold.dim_caracteristicas")

df_result=(
    dffp
    .join(dfz, dffp.zona_id==dfz.zona_id,"inner")
    .join(dfto, dffp.tipo_operacion_id==dfto.tipo_operacion_id,"inner")
    .join(dfc, dffp.caracteristicas_id==dfc.caracteristicas_id,"inner")
    .groupBy(dfz.partido,dfto.tipo_operacion)
    .agg(
        count("*").alias("cantidad_propiedades"),
        avg(col("precio")).alias("precio_promedio"),
        avg(col("precio_por_m2")).alias("precio_m2_promedio")
    )
    .filter(col("cantidad_propiedades")>50)
    .orderBy(col("precio_promedio").desc())
)

#df_result.explain(True)

df_result.show(10)

+---------------+--------------+--------------------+---------------+------------------+
|        partido|tipo_operacion|cantidad_propiedades|precio_promedio|precio_m2_promedio|
+---------------+--------------+--------------------+---------------+------------------+
|  vicente lopez|      alquiler|                1260|  673495.372222|      11640.981405|
|     san isidro|      alquiler|                2153|  577539.954947|       8765.790632|
|     san miguel|      alquiler|                1730|  572855.802312|      11115.651653|
|      ituzaingo|      alquiler|                1054|  567690.036053|       9196.525892|
|almirante brown|      alquiler|                 687|  565685.371179|      13108.424527|
|tres de febrero|      alquiler|                1391|  561120.128684|      11470.451804|
|        quilmes|      alquiler|                1508|  558016.083554|      10747.283780|
|lomas de zamora|      alquiler|                2214|  557667.073171|       9880.532724|
|       ensenada|    

In [ ]:
# E3.4 Window Functions — ranking ROW_NUMBER

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col,desc

dfp=spark.table("bootcamp.gold.fact_propiedades")
dfz=spark.table("bootcamp.gold.dim_zona")
dfto=spark.table("bootcamp.gold.dim_tipo_operacion")

wfunc=(
    Window
    .partitionBy("partido")
    .orderBy(col("precio").desc())
)



df_result=(

    dfp
    .join(dfz, dfp.zona_id==dfz.zona_id, "inner")
    .join(dfto, dfp.tipo_operacion_id==dfto.tipo_operacion_id,"inner")
    .withColumn(
        "rank",
        row_number().over(wfunc)
    )
    .filter(col("rank")<=3)
    .select("row_hash","precio", "partido","tipo_operacion", "rank")
)

df_result.show(20)

+--------------------+----------+---------------+--------------+----+
|            row_hash|    precio|        partido|tipo_operacion|rank|
+--------------------+----------+---------------+--------------+----+
|328b157df611e6454...|2700000.00|almirante brown|      alquiler|   1|
|d1491f5433879e30f...|2300000.00|almirante brown|      alquiler|   2|
|ea85f0df7ca2b2dee...|1800000.00|almirante brown|      alquiler|   3|
|aec34fa998347d94f...|2500000.00|     avellaneda|      alquiler|   1|
|f69b8adbe0e7735c6...|2000000.00|     avellaneda|      alquiler|   2|
|fed63f8dba48447f1...|2000000.00|     avellaneda|      alquiler|   3|
|7be7f3096c0a6927e...|3500000.00|    berazategui|         venta|   1|
|feddc18065103fa8f...|2100000.00|    berazategui|      alquiler|   2|
|82f882b18c317cda5...|2100000.00|    berazategui|      alquiler|   3|
|97915e9289e91fb75...|1390000.00|       canuelas|         venta|   1|
|d922a084cc279c830...| 950000.00|       canuelas|      alquiler|   2|
|5432aba9d8b632c6b..

In [18]:
# E3.4 Window Functions — ranking RANK()

from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col,desc

dfp=spark.table("bootcamp.gold.fact_propiedades")
dfz=spark.table("bootcamp.gold.dim_zona")
dfto=spark.table("bootcamp.gold.dim_tipo_operacion")

wfunc=(
    Window
    .partitionBy("partido")
    .orderBy(col("precio").desc())
)



df_result=(

    dfp
    .join(dfz, dfp.zona_id==dfz.zona_id, "inner")
    .join(dfto, dfp.tipo_operacion_id==dfto.tipo_operacion_id,"inner")
    .withColumn(
        "rank",
        rank().over(wfunc)
    )
    .filter(col("rank")<=3)
    .select("row_hash","precio", "partido","tipo_operacion", "rank")
)

df_result.show(20)

+--------------------+----------+---------------+--------------+----+
|            row_hash|    precio|        partido|tipo_operacion|rank|
+--------------------+----------+---------------+--------------+----+
|328b157df611e6454...|2700000.00|almirante brown|      alquiler|   1|
|d1491f5433879e30f...|2300000.00|almirante brown|      alquiler|   2|
|ea85f0df7ca2b2dee...|1800000.00|almirante brown|      alquiler|   3|
|aec34fa998347d94f...|2500000.00|     avellaneda|      alquiler|   1|
|f69b8adbe0e7735c6...|2000000.00|     avellaneda|      alquiler|   2|
|fed63f8dba48447f1...|2000000.00|     avellaneda|      alquiler|   2|
|7be7f3096c0a6927e...|3500000.00|    berazategui|         venta|   1|
|feddc18065103fa8f...|2100000.00|    berazategui|      alquiler|   2|
|82f882b18c317cda5...|2100000.00|    berazategui|      alquiler|   2|
|01865fab71e54105b...|2100000.00|    berazategui|      alquiler|   2|
|97915e9289e91fb75...|1390000.00|       canuelas|         venta|   1|
|d922a084cc279c830..